In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import os
from sklearn.metrics import f1_score, mean_absolute_error, accuracy_score


class MOCSE_Temporal_Dataset(Dataset):
    def __init__(self, annotations_file, child_pers_file, inst_pers_file, openface_dir, max_frames=300):
        self.annotations = pd.read_csv(annotations_file)

        self.child_pers = pd.read_csv(child_pers_file).set_index('vid_title_name')
        self.inst_pers = pd.read_csv(inst_pers_file).set_index('Instructor Name')
        
        self.openface_dir = openface_dir
        self.max_frames = max_frames
        self.valid_data = []
        
        for _, row in self.annotations.iterrows():
            filename = row['file']
            label = row['engaged']
            
            parts = filename.replace('.mp4', '').split('_')
            child_name = parts[-2]

            if "Catherine" in filename: inst_name = "Catherine"
            elif "Pierre" in filename: inst_name = "Pierre"
            elif "Nigel" in filename: inst_name = "Nigel Lu"
            else: continue
            
            if child_name not in self.child_pers.index or inst_name not in self.inst_pers.index:
                continue

            # np.ravel ensures it stays 1D even if a name is accidentally duplicated in the CSV
            c_traits = np.ravel(self.child_pers.loc[child_name].values)[:10]
            i_traits = np.ravel(self.inst_pers.loc[inst_name].values)[:10]

            combined_personality = np.concatenate([c_traits, i_traits]).astype(np.float32)

            clean_id = filename.replace('.mp4', '')
            openface_csv = os.path.join(self.openface_dir, clean_id, f"{clean_id}.csv")
            
            if os.path.exists(openface_csv):
                self.valid_data.append({
                    'csv_path': openface_csv,
                    'personality': combined_personality,
                    'label': np.float32(label)
                })
                
        print(f"Matched {len(self.valid_data)} video sequences")

    def __len__(self):
        return len(self.valid_data)

    def __getitem__(self, idx):
        item = self.valid_data[idx]

        df_seq = pd.read_csv(item['csv_path'])
        df_seq = df_seq.drop(columns=['frame', 'face_id', 'timestamp', 'confidence', 'success'], errors='ignore')
        
        seq_array = df_seq.values.astype(np.float32)

        if len(seq_array) > self.max_frames:
            seq_array = seq_array[:self.max_frames]
        elif len(seq_array) < self.max_frames:
            padding = np.zeros((self.max_frames - len(seq_array), seq_array.shape[1]), dtype=np.float32)
            seq_array = np.vstack((seq_array, padding))
            
        return torch.tensor(seq_array), torch.tensor(item['personality']), torch.tensor(item['label'])

class PersonalityLSTM(nn.Module):
    def __init__(self, input_features_dim, personality_dim=20, hidden_dim=128):
        super(PersonalityLSTM, self).__init__()

        self.pers_encoder = nn.Linear(personality_dim, hidden_dim)

        # 2-Layer Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=input_features_dim, 
            hidden_size=hidden_dim, 
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )
        
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim * 2, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, seq, pers):
        pers_encoded = self.pers_encoder(pers)
        
        h_0 = pers_encoded.unsqueeze(0).repeat(4, 1, 1)
        c_0 = torch.zeros_like(h_0)

        lstm_out, _ = self.lstm(seq, (h_0, c_0))
        
        final_state = lstm_out[:, -1, :] 
        return self.regressor(final_state).squeeze(-1)
    

def calculate_ccc(y_true, y_pred):
    """Calculates Concordance Correlation Coefficient (CCC)"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    cor = np.corrcoef(y_true, y_pred)[0][1]
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true, var_pred = np.var(y_true), np.var(y_pred)
    sd_true, sd_pred = np.std(y_true), np.std(y_pred)
    numerator = 2 * cor * sd_true * sd_pred
    denominator = var_true + var_pred + (mean_true - mean_pred)**2
    return numerator / denominator

def evaluate(device):
    dataset = MOCSE_Temporal_Dataset("balanced_annotations_train_aggregated.csv", "personality_train.csv", "Instructor_personality.csv", "./balanced_features/video_without_context/train")
    
    sample_seq, _, _ = dataset[0]
    model = PersonalityLSTM(input_features_dim=sample_seq.shape[1]).to(device)
    model.load_state_dict(torch.load("engagement_model.pth"))
    model.eval()

    true_labels = []
    predictions = []
    introversion_scores = []

    print("Evaluating model...")
    with torch.no_grad():
        for i in range(len(dataset)):
            seq, pers, label = dataset[i]
            seq, pers, label = seq.to(device), pers.to(device), label.to(device)
            introversion = pers[0].item() 
            
            pred = model(seq.unsqueeze(0), pers.unsqueeze(0))
            
            true_labels.append(label.item())
            predictions.append(pred.item())
            introversion_scores.append(introversion)

    binary_preds = [1 if p >= 0.5 else 0 for p in predictions]
    binary_trues = [int(t) for t in true_labels]

    mae = mean_absolute_error(true_labels, predictions)
    f1 = f1_score(binary_trues, binary_preds, zero_division=0)
    acc = accuracy_score(binary_trues, binary_preds)
    ccc = calculate_ccc(true_labels, predictions)

    print(f"\n--- OVERALL METRICS ---")
    print(f"Accuracy: {acc:.4f} (Higher is better)")
    print(f"MAE: {mae:.4f} (Lower is better)")
    print(f"F1 Score: {f1:.4f} (Higher is better)")
    print(f"CCC: {ccc:.4f} (Closer to 1 is better)")

    # Split students based on Introversion (e.g., > 3 is highly introverted)
    df_results = pd.DataFrame({
        'true': true_labels, 
        'pred': predictions, 
        'introversion': introversion_scores
    })

    introverts = df_results[df_results['introversion'] > 3]
    extroverts = df_results[df_results['introversion'] <= 3]

    # Convert continuous predictions to binary specifically for the subgroups
    introverts_binary_preds = (introverts['pred'] >= 0.5).astype(int)
    extroverts_binary_preds = (extroverts['pred'] >= 0.5).astype(int)

    print(f"\n--- PERSONALITY SPLIT ANALYSIS ---")
    print(f"Introverts (n={len(introverts)}):")
    print(f"  MAE: {mean_absolute_error(introverts['true'], introverts['pred']):.4f}")
    print(f"  Accuracy: {accuracy_score(introverts['true'], introverts_binary_preds):.4f}")

    print(f"\nExtroverts (n={len(extroverts)}):")
    print(f"  MAE: {mean_absolute_error(extroverts['true'], extroverts['pred']):.4f}")
    print(f"  Accuracy: {accuracy_score(extroverts['true'], extroverts_binary_preds):.4f}")

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    evaluate(device)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import os

class MOCSE_Temporal_Dataset(Dataset):
    def __init__(self, annotations_file, child_pers_file, inst_pers_file, openface_dir, max_frames=300):
        self.annotations = pd.read_csv(annotations_file)

        self.child_pers = pd.read_csv(child_pers_file).set_index('vid_title_name')
        self.inst_pers = pd.read_csv(inst_pers_file).set_index('Instructor Name')
        
        self.openface_dir = openface_dir
        self.max_frames = max_frames
        self.valid_data = []
        
        for _, row in self.annotations.iterrows():
            filename = row['file']
            label = row['engaged']
            
            parts = filename.replace('.mp4', '').split('_')
            child_name = parts[-2]

            if "Catherine" in filename: inst_name = "Catherine"
            elif "Pierre" in filename: inst_name = "Pierre"
            elif "Nigel" in filename: inst_name = "Nigel Lu"
            else: continue
            
            if child_name not in self.child_pers.index or inst_name not in self.inst_pers.index:
                continue

            c_traits = np.ravel(self.child_pers.loc[child_name].values)[:10]
            i_traits = np.ravel(self.inst_pers.loc[inst_name].values)[:10]

            combined_personality = np.concatenate([c_traits, i_traits]).astype(np.float32)

            clean_id = filename.replace('.mp4', '')
            openface_csv = os.path.join(self.openface_dir, clean_id, f"{clean_id}.csv")
            
            if os.path.exists(openface_csv):
                self.valid_data.append({
                    'csv_path': openface_csv,
                    'personality': combined_personality,
                    'label': np.float32(label)
                })
                
        print(f"Matched {len(self.valid_data)} video sequences")

    def __len__(self):
        return len(self.valid_data)

    def __getitem__(self, idx):
        item = self.valid_data[idx]

        df_seq = pd.read_csv(item['csv_path'])
        df_seq = df_seq.drop(columns=['frame', 'face_id', 'timestamp', 'confidence', 'success'], errors='ignore')
        
        seq_array = df_seq.values.astype(np.float32)

        if len(seq_array) > self.max_frames:
            seq_array = seq_array[:self.max_frames]
        elif len(seq_array) < self.max_frames:
            padding = np.zeros((self.max_frames - len(seq_array), seq_array.shape[1]), dtype=np.float32)
            seq_array = np.vstack((seq_array, padding))
            
        return torch.tensor(seq_array), torch.tensor(item['personality']), torch.tensor(item['label'])

class PersonalityLSTM(nn.Module):
    def __init__(self, input_features_dim, personality_dim=20, hidden_dim=128):
        super(PersonalityLSTM, self).__init__()

        self.pers_encoder = nn.Linear(personality_dim, hidden_dim)

        self.lstm = nn.LSTM(
            input_size=input_features_dim, 
            hidden_size=hidden_dim, 
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )
        
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim * 2, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, seq, pers):
        pers_encoded = self.pers_encoder(pers)
        
        h_0 = pers_encoded.unsqueeze(0).repeat(4, 1, 1)
        c_0 = torch.zeros_like(h_0)

        lstm_out, _ = self.lstm(seq, (h_0, c_0))
        
        final_state = lstm_out[:, -1, :] 
        return self.regressor(final_state).squeeze(-1)
    
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    ANNOTATIONS = "balanced_annotations_train_aggregated.csv"
    CHILD_PERS = "personality_train.csv"
    INST_PERS = "Instructor_personality.csv"
    OPENFACE_DIR = "./balanced_features/video_without_context/train" 

    dataset = MOCSE_Temporal_Dataset(ANNOTATIONS, CHILD_PERS, INST_PERS, OPENFACE_DIR)

    if len(dataset) == 0:
        print("No valid data found. Exiting.")
    else:
        sample_seq, _, _ = dataset[0]
        num_features = sample_seq.shape[1]

        model = PersonalityLSTM(input_features_dim=num_features).to(device)

        dataloader = DataLoader(dataset, batch_size=8, shuffle=True)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
        criterion = nn.SmoothL1Loss()

        print(f"Started training!")
        model.train()
        for epoch in range(250):
            total_loss = 0
            correct_preds = 0
            total_samples = 0
            
            for seqs, pers, labels in dataloader:
                seqs, pers, labels = seqs.to(device), pers.to(device), labels.to(device)
                optimizer.zero_grad()
                preds = model(seqs, pers)
                loss = criterion(preds, labels)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                
                # Accuracy tracking: predict 1 if >= 0.5, else 0
                binary_preds = (preds >= 0.5).float()
                correct_preds += (binary_preds == labels).sum().item()
                total_samples += labels.size(0)
                
            epoch_loss = total_loss / len(dataloader)
            epoch_acc = correct_preds / total_samples
            
            print(f"Epoch {epoch+1} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f}")
            
        torch.save(model.state_dict(), "engagement_model.pth")
        print("Done! Saved to engagement_model.pth")